<a href="https://colab.research.google.com/github/gayakarapetyan/PythonCourseH2/blob/main/Session1/Part2_python_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session1 Part 2 — Data Analysis with Python
### Statistics and Machine Learning

---

In Part 1 we learned the Python basics: variables, loops, functions, classes.

In this session we use that foundation to work with real data. We will load atmospheric electric field measurements and weather station data recorded in Magdeburg, explore and clean them, visualize them, and look for relationships between variables.

By the end of this session you will be able to:
- load any CSV file into Python
- explore, filter and clean a dataset
- produce publication-quality plots
- calculate and visualize correlations between variables

---

## 1. Libraries

Python comes with a small set of built-in functions. Everything else lives in **libraries** — collections of code written by other people that you can reuse.

You load a library with `import`. You can give it a shorter alias using `as` — this is just a nickname so you type less.

The four libraries we use today:

| Library | What it does | Usual alias |
|---------|-------------|-------------|
| `numpy` | fast numerical operations on arrays | `np` |
| `pandas` | working with tabular data (like Excel) | `pd` |
| `matplotlib` | creating plots and figures | `plt` |
| `seaborn` | statistical visualizations built on matplotlib | `sns` |

In [ ]:
# Install any libraries not already available in Colab
# Pandas, numpy, matplotlib and seaborn are pre-installed — no need to install them
# Plotly is also pre-installed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

print("All libraries imported successfully")
print(f"Pandas version:  {pd.__version__}")
print(f"NumPy version:   {np.__version__}")

---
## 2. Introduction to Pandas

Pandas is the most important library for data analysis in Python. Its main object is the **DataFrame** — a table with rows and columns, similar to a spreadsheet.

Before we load any real data, we will build a small DataFrame by hand so you understand exactly what it is and how it works.

### 2.1 Creating a DataFrame from scratch

In [ ]:
# We can create a DataFrame from a dictionary
# Each key becomes a column name, each value is a list of column values

# First let us use numpy to generate some random data
# np.random.randint(low, high, size) generates random integers
# np.random.uniform(low, high, size) generates random floats

np.random.seed(42)   # seed makes the random numbers reproducible

data = {
    "day"         : ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    "temperature" : np.random.uniform(10, 50, 7).round(1),
    "humidity"    : np.random.randint(40, 95, 7),
    "wind_speed"  : np.random.uniform(0, 15, 7).round(1)
}

# pd.DataFrame() turns the dictionary into a table
df = pd.DataFrame(data)
print(df)

### 2.2 First things to check about any DataFrame

When you load or create a DataFrame, these are the first things you always want to know.

In [ ]:
# shape returns a tuple (number of rows, number of columns)
print(df.shape)
print(f"Rows: {df.shape[0]}  Columns: {df.shape[1]}")

In [ ]:
# dtypes shows the data type of each column
# object means string, int64 means integer, float64 means decimal number
print(df.dtypes)

In [ ]:
# describe() gives basic statistics for all numeric columns
# count, mean, std (standard deviation), min, 25th percentile, median, 75th percentile, max
print(df.describe())

In [ ]:
# head() shows the first N rows (default 5)
# tail() shows the last N rows
print(df.head(5))
print()
print(df.tail(2))

### 2.3 Column names — accessing and renaming

In [ ]:
# See all column names
print(df.columns)
print(list(df.columns))   # as a plain list

In [ ]:
# Rename columns
# Pass a dictionary: {"old_name": "new_name"}
df = df.rename(columns={"wind_speed": "wind_ms"})
print(df.columns)

### 2.4 Selecting columns

In [ ]:
# Select one column by name — returns a Series (a single column)
temps = df["temperature"]
print(type(temps))   # pandas Series
print(temps)

In [ ]:
# Select multiple columns by name — pass a list of names
# Returns a DataFrame
subset = df[["day", "temperature", "humidity"]]
print(subset)

In [ ]:
# Select columns by index position using .iloc
# .iloc[rows, columns] — use : to mean all rows
print(df.iloc[:, 0])      # first column
print()
print(df.iloc[:, 1:3])    # columns at index 1 and 2

### 2.5 Sorting

In [ ]:
# Sort by a column value
# ascending=False means highest first
df_sorted = df.sort_values("temperature", ascending=False)
print(df_sorted)

### 2.6 A quick plot directly from a DataFrame

Pandas has a built-in `.plot()` method that works for quick exploration. We will use matplotlib properly later — this is just to show the possibility.

In [ ]:
# Quick line plot of temperature
df.plot(x="day", y="temperature", title="Temperature across the week")
plt.show()

---
## 3. Introduction to NumPy

NumPy (Numerical Python) is the foundation of scientific computing in Python. Its main object is the **array** — like a Python list but much faster for mathematical operations, especially on large datasets.

Pandas is built on top of NumPy — every column of a DataFrame is internally a NumPy array.

In [ ]:
# Creating a NumPy array from a plain list
temps_list  = [13.5, 14.2, 12.8, 8.7, 10.2, 12.5, 13.9]
temps_array = np.array(temps_list)

print(type(temps_list))    # plain Python list
print(type(temps_array))   # NumPy array

In [ ]:
# NumPy statistical functions work on both lists and arrays
print(np.mean(temps_list))    # works on plain list
print(np.mean(temps_array))   # works on array
print(np.std(temps_array))    # standard deviation
print(np.median(temps_array)) # median
print(np.percentile(temps_array, 25))   # 25th percentile
print(np.percentile(temps_array, 75))   # 75th percentile

In [ ]:
# NumPy operations apply to every element at once
# No need to loop
print(temps_array * 9/5 + 32)   # convert all to Fahrenheit at once
print(temps_array - np.mean(temps_array))   # subtract mean from every value

In [ ]:
# NumPy functions also work directly on DataFrame columns
print(np.mean(df["temperature"]))
print(np.std(df["temperature"]))

In [ ]:
# Useful NumPy array generators
print(np.zeros(5))              # array of zeros
print(np.ones(5))               # array of ones
print(np.linspace(0, 10, 5))    # 5 evenly spaced values from 0 to 10
print(np.arange(0, 10, 2))      # like range() but returns an array

---
## 4. Reading CSV files

Now we load real data. We have two CSV files:
- **efield.csv** — atmospheric electric field measurements
- **weather.csv** — full weather station data

Both files have no header row, so we assign column names manually.

First we mount Google Drive so Colab can access our files.

In [ ]:
# Mount Google Drive
# This will ask you to authorize access once
# After mounting, your Drive is available at /content/drive/MyDrive/
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PATH_EFIELD  = '/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session1/Data/efm_data_processed.csv'
PATH_WEATHER = '/content/drive/MyDrive/Colab Notebooks/PythonCourse/Session1/Data/weather_data_processed.csv'

In [ ]:
# pd.read_csv() reads a CSV file into a DataFrame
# Why parse_dates? By default pandas reads dates as plain strings.
# When parsed as datetime, you can filter by date, group by hour, etc.

df_ef = pd.read_csv(
    PATH_EFIELD,
    parse_dates=["datetime",],
)

In [ ]:
# Always print shape and columns immediately after loading
# shape = (number of rows, number of columns)
print("Shape:", df_ef.shape)
print()
print("Column names:")
print(list(df_ef.columns))

In [ ]:
# pd.read_csv() reads a CSV file into a DataFrame
# Why parse_dates? By default pandas reads dates as plain strings.
# When parsed as datetime, you can filter by date, group by hour, etc.

#efield_columns = ["id", "datetime", "efield", "col4", "col5", "col6", "col7", "col8"]

'''df_ef = pd.read_csv(
    PATH_EFIELD,
    header=None,
    names=efield_columns,
    parse_dates=["datetime",],
    skiprows =1
)
'''

In [ ]:
# Now look at the first few rows
print(df_ef.head(5))

In [ ]:
# Check data types of each column
# datetime64[ns] means the datetime column was correctly parsed as a timestamp type
# This is important — without parse_dates it would just be object (string)
print(df_ef.dtypes)

In [ ]:
# If for any reason the datetime column was not parsed automatically,
# you can convert it manually with pd.to_datetime()
# pd.to_datetime() converts a string column to the datetime64 type
# The format string tells pandas how to interpret the text
#
# Example format: "2021-03-20 10:32:39" -> "%Y-%m-%d %H:%M:%S"
#
# df_ef["datetime"] = pd.to_datetime(df_ef["datetime"], format="%Y-%m-%d %H:%M:%S")

print(df_ef["datetime"].dtype)

In [ ]:
# Load the weather station data the same way

df_wx = pd.read_csv(PATH_WEATHER, parse_dates=["timestamp"])


In [ ]:
print("Shape:", df_wx.shape)
print()
print("Column names:")
print(list(df_wx.columns))

In [ ]:
# Load the weather station data the same way
'''weather_columns = [
    "id", "timestamp",
    "windspeed", "winddirection", "maxwind", "maxwindangle",
    "airtemperature", "insidetemp", "Akustictemperature", "airtempuncorr",
    "relativehumidity", "absolutehumid", "dewpoint",
    "absolutepressure", "pressure",
    "brightnessN", "brightnessE", "brightnessS", "brightnessW",
    "brightnessdirection", "brightnessmaximum",
    "precipstatus", "precipitation", "precipsum", "preciptype",
    "date", "lenght", "hight", "elevangle", "azimuth", "altitude",
    "voltage", "counter", "error"
]

df_wx = pd.read_csv(
    PATH_WEATHER,
    header=None,
    names=weather_columns,
    parse_dates=["timestamp"]
)

print("Shape:", df_wx.shape)
print()
print("Column names:")
print(list(df_wx.columns))'''

In [ ]:
print(df_wx.head())

In [ ]:
# Basic statistics for numeric columns
print(df_wx[["airtemperature", "relativehumidity", "windspeed", "precipitation"]].describe())

---
## 5. Data selection and filtering

Once we have a DataFrame, we need to select the parts we are interested in. Pandas gives us several ways to do this.

### 5.1 Filtering rows by condition

In [ ]:
# Select only rows where efield is above 400 V/m (storm conditions)
# The condition inside [] returns True/False for each row
storm_rows = df_ef[df_ef["efield"] > 400]
print(f"Total rows:  {len(df_ef)}")
print(f"Storm rows:  {len(storm_rows)}")

In [ ]:
# Combine conditions with & (and) and | (or)
# Note: each condition must be in parentheses when combining
high_humidity = df_wx[
    (df_wx["relativehumidity"] > 80) & (df_wx["airtemperature"] < 15)
]
print(f"High humidity and cold rows: {len(high_humidity)}")

### 5.2 Filtering by date and time

Because our timestamp column is a proper datetime type, we can filter by date ranges very easily.

In [ ]:
# Select data from a specific date range
start = "2026-04-06"
end   = "2026-04-08"

mask   = (df_ef["datetime"] >= start) & (df_ef["datetime"] <= end)
df_two_days = df_ef[mask]

print(f"Rows in selected range: {len(df_two_days)}")
print(df_two_days.head())

In [ ]:
# Select only weekdays (Monday=0 to Friday=4)
# .dt is an accessor for datetime properties on a column
# .dt.dayofweek returns 0=Monday, 1=Tuesday ... 6=Sunday
weekdays = df_ef[df_ef["datetime"].dt.dayofweek < 5]
print(f"Weekday rows: {len(weekdays)}")
print(f"All rows:     {len(df_ef)}")

In [ ]:
#df_ef["datetime"].dt.month

### 5.3 Missing values

Real data always has missing values. Pandas represents them as `NaN` (Not a Number). We need to know how to find them and decide what to do with them.

Our data does not have NaN values, so we will add some artificially to demonstrate.

In [ ]:
# Make a copy so we do not damage the original
df_test = df_ef.copy()

# Introduce NaN values at specific positions
# np.nan is the standard missing value marker
df_test.loc[5, "efield"]  = np.nan
df_test.loc[10, "efield"] = np.nan
df_test.loc[15, "efield"] = np.nan

# Check how many NaN values exist per column
print(df_test.isnull().sum())

In [ ]:
# dropna() removes any row that contains at least one NaN value
# It returns a new DataFrame — it does not modify the original
df_clean = df_test.dropna()
print(f"Before dropna: {len(df_test)} rows")
print(f"After dropna:  {len(df_clean)} rows")

In [ ]:
# Alternative: fill NaN values instead of dropping rows
# fillna() replaces NaN with a value you specify
df_filled = df_test.fillna(df_test["efield"].mean())
print(df_filled.isnull().sum())

### 5.4 Calculations on columns

In [ ]:
# You can apply arithmetic to an entire column at once
# This creates a new column in the DataFrame

# Example: normalize efield by dividing by fair-weather baseline (100 V/m)
df_ef["efield_normalized"] = df_ef["efield"] / 100

# Example: just to show it is possible — multiply and shift
df_ef["efield_scaled"] = df_ef["efield"] * 5 - 100

print(df_ef[["efield", "efield_normalized", "efield_scaled"]].head())

---
## 6. Working with timestamps

Because our datetime column is a proper datetime type, we can extract components from it and group the data by time periods.

The `.dt` accessor gives you access to all datetime properties.

In [ ]:
# Extract components from the datetime column
df_ef["year"]   = df_ef["datetime"].dt.year
df_ef["month"]  = df_ef["datetime"].dt.month
df_ef["day"]    = df_ef["datetime"].dt.day
df_ef["hour"]   = df_ef["datetime"].dt.hour
df_ef["weekday"]= df_ef["datetime"].dt.day_name()   # Monday, Tuesday etc

print(df_ef[["datetime", "year", "month", "day", "hour", "weekday"]].head(10))

In [ ]:
# groupby() splits the data into groups and applies a function to each group
# Here we calculate the mean efield for each hour of the day
hourly_mean = df_ef.groupby("hour")["efield"].mean()
print(hourly_mean)

In [ ]:
# Group by day
daily_mean = df_ef.groupby("day")["efield"].mean()
print(daily_mean)

In [ ]:
# Group by multiple levels
# For example: mean efield per day per hour
day_hour_mean = df_ef.groupby(["day", "hour"])["efield"].mean()
print(day_hour_mean.head(20))

In [ ]:
# Select only a specific weekday
mondays = df_ef[df_ef["weekday"] == "Monday"]
print(f"Monday rows: {len(mondays)}")

---
## 7. Plotting with Matplotlib

Matplotlib is the standard plotting library in Python. It gives you complete control over every element of a figure.

We build plots step by step — starting with the simplest possible plot and adding elements one at a time.

In [ ]:
# Define our color palette — we use these consistently across all plots
BLUE   = "#1565C0"
ORANGE = "#E8431A"

### 7.1 The simplest possible plot

In [ ]:
# Select one day of efield data
# you can do first day by df_ef["day"].iloc[0]  or you can select a date

one_day = df_ef[df_ef['datetime'].dt.date == pd.Timestamp('2026-05-05').date()].copy()

# plt.figure() creates a new figure
# figsize=(width, height) in inches
plt.figure(figsize=(10, 4))
# plt.plot(x, y) draws a line
plt.plot(one_day["datetime"], one_day["efield"])

plt.show()

### 7.2 Adding labels and styling

In [ ]:
# Now we add labels, title, and remove the default grey background and grid

fig, ax = plt.subplots(figsize=(12, 4))
# fig is the whole figure, ax is the plot area inside it
# Using fig, ax gives us more control than plt.plot() alone

ax.plot(one_day["datetime"], one_day["efield"], color=BLUE)

# Labels and title
ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Electric field (V/m)", fontsize=12)
ax.set_title("Atmospheric electric field — Magdeburg", fontsize=14)

# Remove background color and grid
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)

# Keep axis lines but remove the top and right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Control the number of tick marks on the x axis
# MaxNLocator limits the number of ticks so the axis is not crowded
from matplotlib.ticker import MaxNLocator
ax.xaxis.set_major_locator(MaxNLocator(nbins=8))

# Rotate x axis labels so they do not overlap
plt.xticks(rotation=0, ha="right")

plt.tight_layout()
plt.show()

### 7.3 Line styles, markers and legend

In [ ]:
# Plot two lines on the same axes with different styles
# Use a short time window so markers are visible
short = one_day.iloc[:50].copy()

fig, ax = plt.subplots(figsize=(12, 4))

# linestyle options: "-" solid, "--" dashed, ":" dotted, "-." dash-dot
# marker options: "o" circle, "s" square, "^" triangle, "." point
ax.plot(short["datetime"], short["efield"],
        color=BLUE, linestyle="-", marker="o", markersize=4,
        label="E-field")

ax.plot(short["datetime"], short["col5"],
        color=ORANGE, linestyle="--", marker=".", markersize=4,
        label="col5")

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Value", fontsize=12)
ax.set_title("E-field and col5", fontsize=14)
ax.legend(fontsize=11)

ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(MaxNLocator(nbins=6))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 7.4 Saving a figure

In [ ]:
# savefig() saves the figure to a file
# dpi controls resolution (dots per inch) — 150 is good for screen, 300 for print
# bbox_inches="tight" prevents labels from being cut off

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(one_day["datetime"], one_day["efield"], color=BLUE)
ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Electric field (V/m)", fontsize=12)
ax.set_title("Atmospheric electric field — Magdeburg", fontsize=14)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

#plt.savefig("efield_plot.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved")

### 7.5 Subplots — multiple plots in one figure

`plt.subplots(rows, cols)` creates a grid of plot areas. You get back a figure and an array of axes — one per subplot.

Common layouts:
- `plt.subplots(1, 2)` — one row, two columns side by side
- `plt.subplots(2, 1)` — two rows, one column stacked
- `plt.subplots(2, 2)` — two by two grid
- `plt.subplots(1, 3)` — one row, three columns

In [ ]:
# 1 row, 2 columns — efield and temperature side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
# axes is now an array: axes[0] is the left plot, axes[1] is the right

# Left plot: efield
axes[0].plot(one_day["datetime"], one_day["efield"], color=BLUE)
axes[0].set_title("Electric field (V/m)", fontsize=13)
axes[0].set_xlabel("Time")
axes[0].set_facecolor("white")
axes[0].grid(False)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)
axes[0].xaxis.set_major_locator(MaxNLocator(nbins=5))
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha="right")

# Right plot: temperature from weather station for the same day
#one_day_wx = df_wx[df_wx["timestamp"].dt.day == df_wx["timestamp"].dt.day.iloc[0]].copy()
one_day_wx = df_wx[df_wx["timestamp"].dt.date == pd.Timestamp('2026-05-05').date()].copy()
axes[1].plot(one_day_wx["timestamp"], one_day_wx["airtemperature"], color=ORANGE)
axes[1].set_title("Air temperature (C)", fontsize=13)
axes[1].set_xlabel("Time")
axes[1].set_facecolor("white")
axes[1].grid(False)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].xaxis.set_major_locator(MaxNLocator(nbins=5))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")

fig.patch.set_facecolor("white")
plt.tight_layout()
plt.show()

### 7.6 Histograms

In [ ]:
# A histogram shows the distribution of values
# bins controls how many bars the range is divided into
# edgecolor adds a border to each bar so they are separated visually

fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(df_ef["efield"].dropna(), bins=50,
        color=BLUE, edgecolor="white")

ax.set_xlabel("Electric field (V/m)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Distribution of electric field values", fontsize=14)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Two histograms in one figure using subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df_ef["efield"].dropna(), bins=50,
             color=BLUE, edgecolor="white")
axes[0].set_title("Electric field distribution", fontsize=13)
axes[0].set_xlabel("E-field (V/m)")
axes[0].set_ylabel("Count")
axes[0].set_facecolor("white")
axes[0].grid(False)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

axes[1].hist(df_wx["airtemperature"].dropna(), bins=40,
             color=ORANGE, edgecolor="white")
axes[1].set_title("Air temperature distribution", fontsize=13)
axes[1].set_xlabel("Temperature (C)")
axes[1].set_ylabel("Count")
axes[1].set_facecolor("white")
axes[1].grid(False)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)

fig.patch.set_facecolor("white")
plt.tight_layout()
plt.show()

### 7.7 Seaborn — nicer statistical plots with less code

Seaborn is built on top of matplotlib. It produces more polished statistical plots with less code. It is especially useful for distribution plots, heatmaps and pair plots.

In [ ]:
# Seaborn histogram with KDE (kernel density estimate) overlay
# KDE is a smoothed version of the histogram — a probability density curve
fig, ax = plt.subplots(figsize=(8, 4))

sns.histplot(df_ef["efield"].dropna(), bins=50, kde=True,
             color=BLUE, ax=ax)

ax.set_xlabel("Electric field (V/m)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Electric field distribution with KDE", fontsize=14)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 7.8 Task — Plot precipitation

Use the weather station DataFrame to explore precipitation.

In [ ]:
# Task 7.1 — Precipitation analysis
#
# Part 1:
#   Print all unique values in the preciptype column
#   (the types are in German — inspect them first)
#
# Part 2:
#   Select one precipitation type and calculate basic statistics
#   for that type: mean precipsum, max precipsum, count of events
#
# Part 3:
#   Plot precipsum over time as a bar chart or line plot
#   Use our standard style: white background, no grid, BLUE color
#
# your code here


In [ ]:
'''# ================================================
# SOLUTION 7.1
# ================================================

# Part 1: unique precipitation types
print("Unique precipitation types:")
print(df_wx["preciptype"].unique())

# Part 2: statistics per precipitation type
print()
print("Statistics per precipitation type:")
print(df_wx.groupby("preciptype")["precipsum"].agg(["mean", "max", "count"]))

# Part 3: plot precipsum over time
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df_wx["timestamp"], df_wx["precipsum"],
        color=BLUE, linewidth=1)

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("Precipitation sum (mm)", fontsize=12)
ax.set_title("Precipitation over time — Magdeburg", fontsize=14)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()'''

---
## 8. Interactive plots with Plotly

Matplotlib produces static images. Plotly produces interactive charts — you can zoom, pan, hover to see exact values. This is very useful for exploring time series data.

In [ ]:
# Interactive efield plot
fig = px.line(
    df_ef, x="datetime", y="efield",
    title="Atmospheric electric field — Magdeburg",
    labels={"efield": "E-field (V/m)", "datetime": "Time"}
)

fig.update_traces(line_color=BLUE)
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", linewidth=1,
               ticks="outside", tickcolor="black"),
    yaxis=dict(showgrid=False, linecolor="black", linewidth=1,
               ticks="outside", tickcolor="black"),
    font=dict(color="black")
)
fig.show()

In [ ]:
# Plot efield and temperature together
# We need both datasets to cover the same time period
# Use go.Figure() for more control when combining multiple traces

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_ef["datetime"], y=df_ef["efield"],
    name="E-field (V/m)",
    line=dict(color=BLUE, width=1)
))

fig.add_trace(go.Scatter(
    x=df_wx["timestamp"], y=df_wx["airtemperature"],
    name="Air temperature (C)",
    line=dict(color=ORANGE, width=1),
    yaxis="y2"   # plot on a second y axis
))

fig.update_layout(
    title="Electric field and temperature — Magdeburg",
    plot_bgcolor="white",
    paper_bgcolor="white",
    xaxis=dict(showgrid=False, linecolor="black", linewidth=1,
               ticks="outside"),
    yaxis=dict(showgrid=False, linecolor="black", linewidth=1,
               title="E-field (V/m)", ticks="outside"),
    yaxis2=dict(showgrid=False, linecolor=ORANGE, linewidth=1,
                title="Temperature (C)", overlaying="y", side="right"),
    font=dict(color="black"),
    legend=dict(bgcolor="white", bordercolor="black", borderwidth=1)
)
fig.show()

In [ ]:
# Smoothing with a rolling mean
# rolling(window=N) computes a moving average over N consecutive values
#
# window  — how many data points to average together
#           larger window = smoother line but loses detail
# center  — if True, the window is centered on each point
#           if False (default), the window looks back
# min_periods — minimum number of observations in window to produce a result

df_ef["efield_smooth"] = df_ef["efield"].rolling(window=60, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df_ef["datetime"], df_ef["efield"],
        color=BLUE, alpha=0.3, linewidth=0.8, label="raw")
ax.plot(df_ef["datetime"], df_ef["efield_smooth"],
        color=BLUE, linewidth=2, label="smoothed (60-point rolling mean)")

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("E-field (V/m)", fontsize=12)
ax.set_title("Electric field — raw and smoothed", fontsize=14)
ax.legend(fontsize=11)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.xaxis.set_major_locator(MaxNLocator(nbins=8))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

---
## 9. Correlation analysis

Correlation measures how strongly two variables move together. This is one of the most important tools in statistics and the foundation of many machine learning methods.

We will work through correlation step by step:
1. Visual exploration with line plots
2. Calculating the correlation coefficient
3. Scatter plot with regression line
4. Different correlation methods
5. Pairplot for multiple variables
6. Correlation heatmap
7. KDE plot
8. Linear regression as a preview of Session 3

### 9.1 Visual exploration — one day of data

Before calculating any numbers, always look at the data first. We select one day and plot efield and humidity together to see if there is a visual relationship.

In [ ]:
# Select one day from each dataset
# We pick the first available day
first_day = one_day

day_ef = one_day.copy()
day_wx = one_day_wx.copy()

print(f"Day selected: {first_day}")
print(f"E-field rows: {len(day_ef)}")
print(f"Weather rows: {len(day_wx)}")

In [ ]:
# Line plot with markers for one day — efield and humidity
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
# sharex=True means both plots share the same x axis

axes[0].plot(day_ef["datetime"], day_ef["efield"],
             color=BLUE, linewidth=1.5, marker=".", markersize=3,
             label="E-field (V/m)")
axes[0].set_ylabel("E-field (V/m)", fontsize=12)
axes[0].set_title(f"Electric field and humidity — day {first_day}", fontsize=14)
axes[0].legend(fontsize=10)
axes[0].set_facecolor("white")
axes[0].grid(False)
axes[0].spines["top"].set_visible(False)
axes[0].spines["right"].set_visible(False)

axes[1].plot(day_wx["timestamp"], day_wx["relativehumidity"],
             color=ORANGE, linewidth=1.5, marker=".", markersize=3,
             label="Relative humidity (%)")
axes[1].set_ylabel("Humidity (%)", fontsize=12)
axes[1].set_xlabel("Time", fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_facecolor("white")
axes[1].grid(False)
axes[1].spines["top"].set_visible(False)
axes[1].spines["right"].set_visible(False)
axes[1].xaxis.set_major_locator(MaxNLocator(nbins=8))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=45, ha="right")

fig.patch.set_facecolor("white")
plt.tight_layout()
plt.show()

### 9.2 Calculating correlation — three ways

The **Pearson correlation coefficient** (r) measures the linear relationship between two variables.
- r = 1.0 means perfect positive correlation
- r = -1.0 means perfect negative correlation
- r = 0.0 means no linear relationship

We first need to align the two datasets by timestamp so we are comparing the same moments in time.

In [ ]:
# Merge efield and weather data on a common time key
# We round timestamps to the nearest minute first so they match
df_ef["datetime_min"] = df_ef["datetime"].dt.floor("min")
df_wx["timestamp_min"] = df_wx["timestamp"].dt.floor("min")

# pd.merge() joins two DataFrames on a common column — like SQL JOIN
df_merged = pd.merge(
    df_ef[["datetime_min", "efield"]],
    df_wx[["timestamp_min", "relativehumidity", "airtemperature",
            "windspeed", "precipsum"]],
    left_on="datetime_min",
    right_on="timestamp_min"
).dropna()

print(f"Merged rows: {len(df_merged)}")
print(df_merged.head())

In [ ]:
# Option 1 — correlation between two plain Python lists
# np.corrcoef returns a 2x2 matrix
# [0,1] gives the correlation between the two variables

list_ef  = df_merged["efield"].tolist()
list_hum = df_merged["relativehumidity"].tolist()

corr_matrix = np.corrcoef(list_ef, list_hum)
r = corr_matrix[0, 1]
print(f"Option 1 (numpy lists):  r = {r:.4f}")

In [ ]:
# Option 2 — correlation between two DataFrame columns
# pandas .corr() is the simplest approach for two columns

r2 = df_merged["efield"].corr(df_merged["relativehumidity"])
print(f"Option 2 (pandas .corr): r = {r2:.4f}")

In [ ]:
# Option 3 — correlation matrix for multiple columns at once
# Returns a DataFrame where each cell is the correlation between two columns

cols = ["efield", "relativehumidity", "airtemperature", "windspeed"]
corr_df = df_merged[cols].corr()
print(corr_df.round(3))

### 9.3 Different correlation methods

There are three common methods:

| Method | What it measures | When to use |
|--------|-----------------|-------------|
| **Pearson** | linear relationship | both variables are continuous and normally distributed |
| **Spearman** | monotonic relationship (ranks) | non-normal data, outliers present, ordinal data |
| **Kendall** | rank concordance | small samples, many tied values |

In [ ]:
from scipy import stats

ef  = df_merged["efield"]
hum = df_merged["relativehumidity"]

# Pearson — measures linear relationship
r_pearson,  p_pearson  = stats.pearsonr(ef, hum)

# Spearman — measures monotonic relationship using ranks
r_spearman, p_spearman = stats.spearmanr(ef, hum)

# Kendall — measures rank concordance
r_kendall,  p_kendall  = stats.kendalltau(ef, hum)

# p-value tells us how likely we would see this result by chance
# p < 0.05 is generally considered statistically significant

print(f"Pearson:  r = {r_pearson:.4f}  p = {p_pearson:.4e}")
print(f"Spearman: r = {r_spearman:.4f}  p = {p_spearman:.4e}")
print(f"Kendall:  r = {r_kendall:.4f}  p = {p_kendall:.4e}")

In [ ]:
df_merged

### 9.4 Scatter plot with correlation line

In [ ]:
# Select one day of merged data for the scatter plot
day_merged = df_merged[df_merged["datetime_min"].dt.date == pd.Timestamp('2026-05-05').date()].copy().reset_index(drop=True)

# Calculate correlation line using numpy polyfit
# polyfit(x, y, degree) fits a polynomial
# degree=1 means a straight line: y = m*x + b
m, b = np.polyfit(day_merged["efield"], day_merged["relativehumidity"], 1)
x_line = np.linspace(day_merged["efield"].min(), day_merged["efield"].max(), 100)
y_line = m * x_line + b

# Calculate correlation coefficients for the legend
r_p, _ = stats.pearsonr(day_merged["efield"], day_merged["relativehumidity"])
r_s, _ = stats.spearmanr(day_merged["efield"], day_merged["relativehumidity"])
r_k, _ = stats.kendalltau(day_merged["efield"], day_merged["relativehumidity"])

fig, ax = plt.subplots(figsize=(7, 6))

# Scatter points
ax.scatter(day_merged["efield"], day_merged["relativehumidity"],
           color=BLUE, alpha=0.5, s=20, label="observations")

# Correlation line
ax.plot(x_line, y_line, color=ORANGE, linewidth=2,
        label=f"Pearson r={r_p:.2f}  Spearman r={r_s:.2f}  Kendall r={r_k:.2f}")

ax.set_xlabel("E-field (V/m)", fontsize=12)
ax.set_ylabel("Relative humidity (%)", fontsize=12)
ax.set_title(f"E-field vs humidity — day {first_day}", fontsize=14)
ax.legend(fontsize=9, loc="upper right")
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout(rect=[0,0,1,0.95])
plt.show()

### 9.5 Pairplot

A pairplot shows scatter plots for every combination of selected variables at once. Very useful for a quick overview of all relationships in a dataset.

In [ ]:
# Pairplot using seaborn
# Use a sample if the dataset is large — pairplot can be slow with many rows
sample = df_merged[["efield", "relativehumidity", "airtemperature", "windspeed"]].sample(
    n=min(500, len(df_merged)), random_state=42
)

g = sns.pairplot(sample, plot_kws={"alpha": 0.4, "color": BLUE},
                 diag_kws={"color": BLUE})
g.figure.suptitle("Pairplot — efield, humidity, temperature, wind",
                  y=1.02, fontsize=13)
plt.show()

### 9.6 KDE plot — density visualization

A KDE (Kernel Density Estimate) plot shows where values are concentrated in two dimensions. Instead of individual points, it shows a smooth density surface — darker areas mean more data points.

In [ ]:
# 2D KDE plot — shows the joint distribution of efield and humidity
fig, ax = plt.subplots(figsize=(7, 6))

sns.kdeplot(
    data=day_merged,
    x="efield",
    y="relativehumidity",
    fill=True,
    cmap="Blues",
    ax=ax
)

ax.set_xlabel("E-field (V/m)", fontsize=12)
ax.set_ylabel("Relative humidity (%)", fontsize=12)
ax.set_title(f"Joint density — E-field vs humidity (day {first_day})", fontsize=14)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 9.7 Correlation heatmap

In [ ]:
# Correlation heatmap — shows all pairwise correlations as colors
# We select a meaningful set of columns, avoiding duplicates

heatmap_cols = ["efield", "relativehumidity", "airtemperature",
                "windspeed", "precipsum"]

corr_matrix = df_merged[heatmap_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))

# annot=True shows the correlation value inside each cell
# fmt=".2f" formats the number to 2 decimal places
# cmap="coolwarm" uses blue for negative, red for positive correlation
sns.heatmap(
    corr_matrix,
    annot=True, fmt=".2f",
    cmap="coolwarm",
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax
)

ax.set_title("Correlation heatmap — weather and efield variables", fontsize=13)
fig.patch.set_facecolor("white")
plt.tight_layout()
plt.show()

### 9.8 Linear regression — preview of Session 3

Correlation tells us the strength of a relationship. Linear regression goes one step further — it gives us a model that can predict one variable from another.

We use scikit-learn here. In Session 3 we will go much deeper into this.

In [ ]:
from sklearn.linear_model import LinearRegression

# Prepare data
# X must be a 2D array for sklearn — reshape(-1,1) converts a 1D array
X = day_merged["relativehumidity"].values.reshape(-1, 1)
y = day_merged["efield"].values

# Fit the model
model = LinearRegression()
model.fit(X, y)

# R-squared score — proportion of variance explained by the model
# 1.0 = perfect, 0.0 = no better than the mean
r2_score = model.score(X, y)

print(f"Slope:     {model.coef_[0]:.4f}")
print(f"Intercept: {model.intercept_:.4f}")
print(f"R-squared: {r2_score:.4f}")
print()
print("Interpretation:")
print(f"For every 1% increase in humidity,")
print(f"efield changes by {model.coef_[0]:.2f} V/m")

In [ ]:
# Plot the regression result
x_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_pred  = model.predict(x_range)

fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(X, y, color=BLUE, alpha=0.4, s=20, label="observations")
ax.plot(x_range, y_pred, color=ORANGE, linewidth=2,
        label=f"Linear regression  R²={r2_score:.2f}")

ax.set_xlabel("Relative humidity (%)", fontsize=12)
ax.set_ylabel("E-field (V/m)", fontsize=12)
ax.set_title("Linear regression: humidity predicts efield", fontsize=14)
ax.legend(fontsize=10)
ax.set_facecolor("white")
fig.patch.set_facecolor("white")
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### 9.9 Task — Correlation analysis

Now it is your turn.

In [ ]:
# Task 9.1 — Explore a different variable pair
#
# Choose a different pair of variables from the merged dataset
# (for example: efield and airtemperature, or windspeed and efield)
#
# 1. Plot both variables over time for one day (line plot with markers)
# 2. Calculate Pearson, Spearman and Kendall correlation
# 3. Make a scatter plot with the regression line
# 4. Write 2-3 sentences interpreting the result:
#    - Is there a relationship?
#    - Is it positive or negative?
#    - Is the R-squared high or low — what does that mean?
#
# your code here


---
## Session 2 complete

Today you covered:

- Importing and using libraries
- Pandas — creating DataFrames, selecting, filtering, sorting, calculations
- NumPy — arrays, statistical functions
- Reading CSV files from Google Drive
- Working with datetime columns
- Matplotlib — step by step from simple to styled plots
- Subplots, histograms, seaborn
- Interactive plots with Plotly
- Correlation — three ways to calculate, three methods, scatter, pairplot, KDE, heatmap
- Linear regression as a preview

In Session 3 we go deeper into machine learning — train/test split, model evaluation, random forests, and predictions on your real data.

---